In [1]:
%load_ext autoreload
%autoreload 2

In [26]:
import os
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch_geometric.data import Data, separate, InMemoryDataset
from torch_geometric.utils import to_dense_batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GATConv, GraphConv
from torch.utils.data import IterableDataset
from torch_geometric.datasets import TUDataset

In [159]:
def load_dataset(path):
    ecs = []
    eos = []

    for subject in tqdm(os.listdir(path)):
        ec = torch.load(os.path.join(path, subject, 'EC_geometric.pt'))
        eo = torch.load(os.path.join(path, subject, 'EO_geometric.pt'))

        ec_indices = len(ec[0].x)
        eo_indices = len(eo[0].x)


        # Break up collated data into individual samples
        # Note that separate is undocumented and I'm not sure what it precisely does
        # but it seems to work and satisfy our use case 
        for i in range(ec_indices):
            ecs.append(separate.separate(cls=ec[0].__class__, batch=ec[0], idx=i, slice_dict=ec[1], decrement=False))

        if eo[1] is None:
            eos.append(eo[0])
        else:
            for i in range(eo_indices):
                eos.append(separate.separate(cls=eo[0].__class__, batch=eo[0], idx=i, slice_dict=eo[1], decrement=False))
    return ecs + eos

In [160]:
class CustomDataset(InMemoryDataset):
    def __init__(self, listOfDataObjects):
        super().__init__()
        self.data, self.slices = self.collate(listOfDataObjects)
    
    def __len__(self):
        return len(self.slices)
    
    def __getitem__(self, idx):
        sample = self.get(idx)
        return sample

In [161]:
datasetList = load_dataset('/home/azureuser/mycontainer/SmallSubset')
print(len(datasetList))

  0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           | 0/20 [00:00<?, ?it/s]

/tmp/ipykernel_87875/2997032793.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ec = torch.load(os.path.join(path, subject, 'EC_geometric.pt'))
/tmp/ipykernel_87875/2997

80


In [162]:
dataset = CustomDataset(datasetList)

In [163]:
loader = DataLoader(datasetList, batch_size=4, shuffle=True)

/anaconda/envs/EEG-ML/lib/python3.8/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [164]:
print(datasetList[0])
print(loader.dataset[0])
print(loader.dataset[1])
print(loader.dataset[2])
print(loader.dataset[3])
print(loader.dataset[4])
print(loader.dataset[5])
print(loader.dataset[6])
print(len(loader.dataset))
out = next(iter(loader))
print(out)
print(out.edge_index.min(), out.edge_index.max())  

Data(x=[19, 15360], edge_index=[2, 361], edge_attr=[19, 19], y=[0], pos=[19, 3])
Data(x=[19, 15360], edge_index=[2, 361], edge_attr=[19, 19], y=[0], pos=[19, 3])
Data(x=[19, 15360], edge_index=[2, 361], edge_attr=[19, 19], y=[0], pos=[19, 3])
Data(x=[19, 15360], edge_index=[2, 361], edge_attr=[19, 19], y=[0], pos=[19, 3])
Data(x=[19, 15360], edge_index=[2, 361], edge_attr=[19, 19], y=[0], pos=[19, 3])
Data(x=[19, 15360], edge_index=[2, 361], edge_attr=[19, 19], y=[0], pos=[19, 3])
Data(x=[19, 15360], edge_index=[2, 361], edge_attr=[19, 19], y=[0], pos=[19, 3])
Data(x=[19, 15360], edge_index=[2, 361], edge_attr=[19, 19], y=[0], pos=[19, 3])
80
DataBatch(x=[4], edge_index=[2, 1444], edge_attr=[4], y=[4], pos=[76, 3], batch=[76], ptr=[5])
tensor(0) tensor(75)


In [165]:
batch = next(iter(loader))
print(batch.x[0].shape)

(19, 15360)


In [189]:
class GNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(in_channels=15360, out_channels=64)

    def forward(self, data):
        x, edge_index, edge_weight = torch.tensor(data.x), data.edge_index, data.edge_attr
        x = torch.flatten(x, start_dim=0, end_dim=1).float()
        print(type(x), type(edge_index))
        print(x.dtype, edge_index.shape)
        print(edge_index.min(), edge_index.max())
        x = self.conv1(x, edge_index)
        return x
    
GNN_module = GNN()

In [190]:
for data in loader:
    print(data)
    print(data.edge_index.min(), data.edge_index.max())
    print(data.edge_index)
    # convert list to tensor
    output = GNN_module(data)
    #print(output.shape)
    break

DataBatch(x=[4], edge_index=[2, 1444], edge_attr=[4], y=[4], pos=[76, 3], batch=[76], ptr=[5])
tensor(0) tensor(75)
tensor([[ 0,  0,  0,  ..., 75, 75, 75],
        [ 0,  1,  2,  ..., 73, 74, 75]])
<class 'torch.Tensor'> <class 'torch.Tensor'>
torch.float32 torch.Size([2, 1444])
tensor(0) tensor(75)
